# St. Paul LVT Shift — HF 1342

**HF 1342** (and its identical senate companion) authorizes Minnesota cities to create **land-value taxation districts** that reallocate the **entire property tax bill** — city, county, school, areawide, state general levy — among parcels based on land value.

### How it works (Section 4)

The city designates a district. Within that district, **every jurisdiction's levy is still collected in full** — the total tax base doesn't change. But the *allocation among parcels* is shifted: land value carries a higher share, improvements carry a lower share.

### Data

Parcel data is scraped from the [Ramsey County ArcGIS FeatureServer](https://maps.co.ramsey.mn.us/arcgis/rest/services/OpenData/OpenData/FeatureServer) and cached locally as geoparquet. If no local file is found, the notebook scrapes fresh data automatically.

Tax statements can be verified at [Ramsey County Beacon](https://beacon.schneidercorp.com/application.aspx?app=RamseyCountyMN&PageType=Search).

In [ ]:
# Step 1: Load parcel data
import sys
import pandas as pd
import geopandas as gpd
import numpy as np
import os
from datetime import datetime

sys.path.insert(0, '../..')

from lvt.lvt_utils import (
    ensure_geodataframe, model_split_rate_tax,
    calculate_category_tax_summary, print_category_tax_summary
)
from lvt.cloud_utils import get_feature_data_with_geometry
from lvt.policy_analysis import (
    analyze_vacant_land, analyze_land_by_improvement_share
)
from lvt.census_utils import (
    get_census_data_with_boundaries, match_to_census_blockgroups,
    calculate_median_percentage_by_quintile
)

pd.set_option('display.max_columns', None)

save_dir = 'data'
os.makedirs(save_dir, exist_ok=True)

# Find most recent geoparquet, scrape if none found
files = [f for f in os.listdir(save_dir) if f.startswith('ramsey_county_') and f.endswith('.gpq')]
files_with_dates = []
for fname in files:
    try:
        date_str = fname.replace('ramsey_county_', '').replace('.gpq', '')
        dt = datetime.strptime(date_str, '%Y%m%d')
        files_with_dates.append((dt, fname))
    except Exception:
        continue

if files_with_dates:
    latest_fname = max(files_with_dates, key=lambda x: x[0])[1]
    fpath = os.path.join(save_dir, latest_fname)
    print(f'Loading: {fpath}')
    ramsey_county_gdf = gpd.read_parquet(fpath)
else:
    print('No local data found — scraping from Ramsey County ArcGIS FeatureServer...')
    base_url = 'https://maps.co.ramsey.mn.us/arcgis/rest/services/OpenData/OpenData/FeatureServer'
    layer_id = 12
    dataset_name = '12/query'
    ramsey_county_gdf = get_feature_data_with_geometry(dataset_name, base_url, layer_id, paginate=True)
    ramsey_county_gdf = ensure_geodataframe(ramsey_county_gdf)
    today_str = datetime.now().strftime('%Y%m%d')
    fpath = os.path.join(save_dir, f'ramsey_county_{today_str}.gpq')
    ramsey_county_gdf.to_parquet(fpath)
    print(f'Saved: {fpath}')

print(f'Loaded {len(ramsey_county_gdf):,} parcels')

## Step 2: Validate Against Official Tax Base

In [ ]:
# Load official tax data from MN Dept of Revenue (optional validation)
# Download: https://www.revenue.state.mn.us/property-tax-history-data

excel_path = os.path.join('data', 'data-portal-excel.xlsx')
has_official_data = os.path.exists(excel_path)

if has_official_data:
    city_town_df = pd.read_excel(excel_path, sheet_name='CityTown', header=0)

    _matches = city_town_df[
        (city_town_df.iloc[:, 5] == 'ST PAUL CITY OF') & 
        (city_town_df.iloc[:, 1] == 2025)
    ]
    if len(_matches) == 0:
        print('Warning: No 2025 St. Paul row found in official data — skipping validation.')
        has_official_data = False
    else:
        st_paul_official = _matches.iloc[0]
        official_data = {
            'Estimated Market Value Total': st_paul_official['Estimated Market Value Total'],
            'Taxable Market Value Total': st_paul_official['Taxable Market Value Total'],
            'Local NTC Total (Tax Capacity)': st_paul_official['Local NTC by Class Total'],
            'TIF NTC': st_paul_official['TIF NTC'],
            'Fiscal Disparities NTC': st_paul_official['Fiscal Disparities Cont NTC'],
            'Taxable NTC': st_paul_official['Taxable NTC'],
            'Net Tax Total': st_paul_official['Net Tax Total'],
            'Total Avg Local NTC Tax Rate': st_paul_official['Total Avg Local NTC Tax Rate'],
            'City/Town Avg Tax Rate': st_paul_official['City/Town Avg Local NTC Tax Rate'],
            'TIF Levy': st_paul_official['TIF Levy'],
        }
        print('OFFICIAL ST. PAUL 2025 TAX DATA (MN Dept of Revenue)')
        for key, val in official_data.items():
            if isinstance(val, float) and val < 10:
                print(f'{key}: {val:.4f}')
            else:
                print(f'{key}: ${val:,.0f}')
        OFFICIAL_TOTAL_NTC_RATE = official_data['Total Avg Local NTC Tax Rate']
        OFFICIAL_CITY_TAX_RATE = official_data['City/Town Avg Tax Rate']
        OFFICIAL_TAXABLE_NTC = official_data['Taxable NTC']
        OFFICIAL_TIF_NTC = official_data['TIF NTC']
        OFFICIAL_LOCAL_NTC = official_data['Local NTC Total (Tax Capacity)']
        OFFICIAL_NET_TAX = official_data['Net Tax Total']
        print(f"\nFull combined NTC rate: {OFFICIAL_TOTAL_NTC_RATE:.4f}")
        print(f"City-only NTC rate:    {OFFICIAL_CITY_TAX_RATE:.4f}")
        print(f"City share of NTC rate: {OFFICIAL_CITY_TAX_RATE / OFFICIAL_TOTAL_NTC_RATE * 100:.1f}%")
else:
    print('data-portal-excel.xlsx not found — skipping official validation.')
    print('Download from: https://www.revenue.state.mn.us/property-tax-history-data')

## Step 3: Filter to St. Paul City-Taxable Parcels

Under HF 1342, the full tax bill is reallocated. We use `TotalTax1` as the current tax for each parcel.

In [ ]:
# Filter to St. Paul parcels
st_paul_gdf = ramsey_county_gdf[ramsey_county_gdf['SiteCityName'] == 'SAINT PAUL'].copy()
print(f'Total St. Paul parcels: {len(st_paul_gdf):,}')

# Flag TIF and exempt parcels
st_paul_gdf['in_tif'] = (
    st_paul_gdf['TIFDistrict'].notna() & 
    (st_paul_gdf['TIFDistrict'].str.strip() != '') &
    (st_paul_gdf['TIFDistrict'] != 'None')
)
st_paul_gdf['fully_exempt'] = (st_paul_gdf['TaxExemptYN'] == 'Y')
st_paul_gdf['pays_city_tax'] = (
    ~st_paul_gdf['in_tif'] & 
    ~st_paul_gdf['fully_exempt'] &
    st_paul_gdf['TaxCapacity'].notna() &
    (st_paul_gdf['TaxCapacity'] > 0)
)

print(f'  In TIF districts:   {st_paul_gdf["in_tif"].sum():,}')
print(f'  Fully tax exempt:   {st_paul_gdf["fully_exempt"].sum():,}')
print(f'  City-taxable:       {st_paul_gdf["pays_city_tax"].sum():,}')

In [ ]:
# Validate scraped data against official tax base
if has_official_data:
    scraped_total_ntc = st_paul_gdf['TaxCapacity'].sum()
    scraped_tif_ntc = st_paul_gdf[st_paul_gdf['in_tif']]['TaxCapacity'].sum()
    scraped_city_taxable_ntc = st_paul_gdf[st_paul_gdf['pays_city_tax']]['TaxCapacity'].sum()
    scraped_total_emv = st_paul_gdf['EMVTotal1'].sum()

    print('VALIDATION: SCRAPED vs OFFICIAL')
    print(f"{'Metric':<30} {'Scraped':>18} {'Official':>18} {'Diff %':>10}")
    print('-' * 80)

    diff_pct = (scraped_total_ntc - OFFICIAL_LOCAL_NTC) / OFFICIAL_LOCAL_NTC * 100
    print(f"{'Total Tax Capacity':<30} ${scraped_total_ntc:>15,.0f} ${OFFICIAL_LOCAL_NTC:>15,.0f} {diff_pct:>9.1f}%")

    diff_pct = (scraped_tif_ntc - OFFICIAL_TIF_NTC) / OFFICIAL_TIF_NTC * 100
    print(f"{'TIF Tax Capacity':<30} ${scraped_tif_ntc:>15,.0f} ${OFFICIAL_TIF_NTC:>15,.0f} {diff_pct:>9.1f}%")

    diff_pct = (scraped_city_taxable_ntc - OFFICIAL_TAXABLE_NTC) / OFFICIAL_TAXABLE_NTC * 100
    print(f"{'City-Taxable NTC':<30} ${scraped_city_taxable_ntc:>15,.0f} ${OFFICIAL_TAXABLE_NTC:>15,.0f} {diff_pct:>9.1f}%")

    diff_pct = (scraped_total_emv - official_data['Estimated Market Value Total']) / official_data['Estimated Market Value Total'] * 100
    print(f"{'Total EMV':<30} ${scraped_total_emv:>15,.0f} ${official_data['Estimated Market Value Total']:>15,.0f} {diff_pct:>9.1f}%")
else:
    print('Skipping validation (no official data loaded)')

In [ ]:
# Create working dataset: city-taxable parcels with FULL tax bill
st_paul_city = st_paul_gdf[st_paul_gdf['pays_city_tax']].copy()

# FULL BILL: current_tax = TotalTax1 (all jurisdictions)
st_paul_city['current_tax'] = st_paul_city['TotalTax1']

current_revenue = st_paul_city['current_tax'].sum()
implied_full_rate = current_revenue / st_paul_city['TaxCapacity'].sum()

print('FULL-BILL TAX BASE')
print(f'Parcels:            {len(st_paul_city):,}')
print(f'Tax Capacity:       ${st_paul_city["TaxCapacity"].sum():,.0f}')
print(f'Total Tax Revenue:  ${current_revenue:,.0f}')
print(f'Implied full rate:  {implied_full_rate:.4f}')
if has_official_data:
    print(f'Official Net Tax:   ${OFFICIAL_NET_TAX:,.0f}')
    print(f'Official NTC rate:  {OFFICIAL_TOTAL_NTC_RATE:.4f}')
print(f'\nCompare to city-only: ${current_revenue * 0.3572:,.0f} (city ~35.7% of full bill)')

## Step 3b: Collapse Condo Units into Buildings

Ramsey County assigns condo units a **token land value** ($1,000 for residential, $500 for garages, $100 for storage). This means the assessor's EMVLand1 for condos is not a real market value — 99.5% of residential condo units show exactly $1,000.

Under the split-rate model, this fake 98.7% improvement ratio would give condos an unearned tax cut. We fix this by:

1. **Collapsing** individual condo units into buildings (grouped by `PlatID`)
2. **Summing** EMVTotal1, TaxCapacity, and TotalTax1 across all units in a building
3. **Imputing** land/building EMV split using the neighborhood's median improvement ratio from non-condo parcels

This gives each condo building a land share that reflects actual neighborhood land values.

In [ ]:
from shapely.ops import unary_union
from shapely.geometry import MultiPolygon

condo_mask = st_paul_city['LandUseCodeDescription'].str.contains('CONDO', case=False, na=False)
condos = st_paul_city[condo_mask].copy()
non_condos = st_paul_city[~condo_mask].copy()

print(f'Before collapse: {len(st_paul_city):,} parcels ({len(condos):,} condo units)')
print(f'  Condo EMVLand1:     ${condos["EMVLand1"].sum():>12,.0f}  (token values)')
print(f'  Condo EMVBuilding1: ${condos["EMVBuilding1"].sum():>12,.0f}')
print(f'  Condo EMVTotal1:    ${condos["EMVTotal1"].sum():>12,.0f}')
print(f'  Condo IR: {condos["EMVBuilding1"].sum() / condos["EMVTotal1"].sum() * 100:.1f}%')

def collapse_geometries(geoms):
    geoms = [g for g in geoms if g is not None]
    if not geoms:
        return None
    out = []
    while geoms:
        ref = geoms.pop(0)
        group = [ref]
        rest = []
        for g in geoms:
            if ref.intersects(g) or ref.touches(g) or ref.equals(g):
                group.append(g)
            else:
                rest.append(g)
        unioned = unary_union(group)
        out.append(unioned)
        geoms = rest
    if len(out) == 1:
        return out[0]
    polygons = []
    for g in out:
        if g.geom_type == "Polygon":
            polygons.append(g)
        elif g.geom_type == "MultiPolygon":
            polygons.extend(g.geoms)
        else:
            polygons.append(g)
    return MultiPolygon(polygons)

subset_cols = ['PlatID']
sum_cols = ['EMVLand1', 'EMVBuilding1', 'EMVTotal1', 'TaxCapacity', 'TotalTax1', 'current_tax']
sum_cols = [c for c in sum_cols if c in condos.columns]

# --- Option A: Collapse with "first" for all values (standard dmp approach) ---
# In jurisdictions where the assessor duplicates the full parcel land value on each
# unit, "first" captures the real value. In St. Paul, land values are $1,000 tokens,
# so this loses total EMV and preserves the fake IR.
agg_dict_a = {col: 'first' for col in condos.columns if col not in (subset_cols + ['geometry'])}
agg_dict_a['geometry'] = collapse_geometries
condos_option_a = condos.groupby(subset_cols, dropna=False).agg(agg_dict_a).reset_index()
condos_option_a = gpd.GeoDataFrame(condos_option_a, geometry='geometry', crs=condos.crs)

print(f'\nOption A (first-value collapse):')
print(f'  {len(condos_option_a):,} buildings')
print(f'  EMVLand1:     ${condos_option_a["EMVLand1"].sum():>12,.0f}  (first unit token)')
print(f'  EMVBuilding1: ${condos_option_a["EMVBuilding1"].sum():>12,.0f}')
print(f'  EMVTotal1:    ${condos_option_a["EMVTotal1"].sum():>12,.0f}  (loses ${condos["EMVTotal1"].sum() - condos_option_a["EMVTotal1"].sum():,.0f})')
print(f'  IR: {condos_option_a["EMVBuilding1"].sum() / condos_option_a["EMVTotal1"].sum() * 100:.1f}%')

# --- Option B (used downstream): Collapse with summed values + imputed land/building ---
# Sum financial columns, impute EMVLand1/EMVBuilding1 from neighborhood large
# multi-family median IR (closest structural comp for condo towers)
agg_dict_b = {col: 'first' for col in condos.columns if col not in (subset_cols + ['geometry'])}
for col in sum_cols:
    agg_dict_b[col] = 'sum'
agg_dict_b['geometry'] = collapse_geometries

condos_collapsed = condos.groupby(subset_cols, dropna=False).agg(agg_dict_b).reset_index()
condos_collapsed = gpd.GeoDataFrame(condos_collapsed, geometry='geometry', crs=condos.crs)

# Spatial join non-condos to District Councils for neighborhood IR
dc_dir = os.path.join('data', 'District_Councils_3386664414246726701')
dc_shp_files = [f for f in os.listdir(dc_dir) if f.endswith('.shp')] if os.path.isdir(dc_dir) else []
if dc_shp_files:
    dc_gdf = gpd.read_file(os.path.join(dc_dir, dc_shp_files[0])).to_crs(epsg=3857)
    nbhd_col = 'councilnam'
else:
    print('Local shapefile not found — fetching District Councils from ArcGIS...')
    dc_gdf = get_feature_data_with_geometry(
        'Saint_Paul_District_Councils',
        'https://services5.arcgis.com/09i2I0fNSnHEF5Ef/arcgis/rest/services',
        layer_id=0, paginate=True
    )
    dc_gdf = ensure_geodataframe(dc_gdf).to_crs(epsg=3857)
    nbhd_col = 'councilname'

nc_pts = non_condos.to_crs(epsg=3857).copy()
nc_pts['_centroid'] = nc_pts.geometry.centroid
nc_pts_gdf = gpd.GeoDataFrame(nc_pts, geometry='_centroid', crs='EPSG:3857')
nc_joined = gpd.sjoin(nc_pts_gdf, dc_gdf[['geometry', nbhd_col]], how='left', predicate='within')
nc_joined['_ir'] = np.where(nc_joined['EMVTotal1'] > 0, nc_joined['EMVBuilding1'] / nc_joined['EMVTotal1'], 0)

# Use large multi-family IR as the comp for condos
lmf_descs = ['APARTMENTS 7-19', 'APARTMENTS 20-49', 'APARTMENTS 50-99',
             'APT OR COMPLEX 100+', 'ASSISTED LIVING', 'NURSING HOME']
lmf_mask = nc_joined['LandUseCodeDescription'].str.upper().apply(
    lambda d: any(x in d for x in lmf_descs)
)
lmf_parcels = nc_joined[lmf_mask]
neighborhood_lmf_ir = lmf_parcels.groupby(nbhd_col)['_ir'].median()
overall_lmf_ir = lmf_parcels['_ir'].median()

print(f'\nLarge multi-family median IR by neighborhood (comp for condos):')
for name, ir in neighborhood_lmf_ir.sort_values().items():
    print(f'  {name:35s} {ir:.3f}  (land share: {1-ir:.1%})')
print(f'  {"OVERALL":35s} {overall_lmf_ir:.3f}  (land share: {1-overall_lmf_ir:.1%})')

# Join collapsed condos to neighborhoods and impute using large MF IR
condo_pts = condos_collapsed.to_crs(epsg=3857).copy()
condo_pts['_centroid'] = condo_pts.geometry.centroid
condo_pts_gdf = gpd.GeoDataFrame(condo_pts, geometry='_centroid', crs='EPSG:3857')
condo_joined = gpd.sjoin(condo_pts_gdf, dc_gdf[['geometry', nbhd_col]], how='left', predicate='within')
if condo_joined.index.duplicated().any():
    condo_joined = condo_joined[~condo_joined.index.duplicated(keep='first')]

condo_joined['_nbhd_ir'] = condo_joined[nbhd_col].map(neighborhood_lmf_ir).fillna(overall_lmf_ir)
condos_collapsed['EMVLand1'] = (condo_joined['EMVTotal1'] * (1 - condo_joined['_nbhd_ir'])).round(0).values
condos_collapsed['EMVBuilding1'] = (condo_joined['EMVTotal1'] * condo_joined['_nbhd_ir']).round(0).values

print(f'\nOption B (summed values + large MF IR) — used downstream:')
print(f'  {len(condos_collapsed):,} buildings')
print(f'  EMVLand1:     ${condos_collapsed["EMVLand1"].sum():>12,.0f}  (imputed from large MF IR)')
print(f'  EMVBuilding1: ${condos_collapsed["EMVBuilding1"].sum():>12,.0f}')
print(f'  EMVTotal1:    ${condos_collapsed["EMVTotal1"].sum():>12,.0f}  (preserved)')
print(f'  IR: {condos_collapsed["EMVBuilding1"].sum() / condos_collapsed["EMVTotal1"].sum() * 100:.1f}%')

# Reassemble st_paul_city using Option B
st_paul_city = pd.concat([non_condos, condos_collapsed], ignore_index=True)
st_paul_city = gpd.GeoDataFrame(st_paul_city, geometry='geometry', crs=condos.crs)
current_revenue = st_paul_city['current_tax'].sum()

print(f'\nAfter collapse: {len(st_paul_city):,} parcels')
print(f'  Non-condo IR: {non_condos["EMVBuilding1"].sum() / non_condos["EMVTotal1"].sum() * 100:.1f}%')
print(f'  Overall IR:   {st_paul_city["EMVBuilding1"].sum() / st_paul_city["EMVTotal1"].sum() * 100:.1f}%')
print(f'  Total revenue: ${current_revenue:,.0f} (unchanged)')


## Step 4: Categorize Properties

In [ ]:
def categorize_st_paul_property(row):
    desc = str(row.get('LandUseCodeDescription', '')).upper()
    use_type = str(row.get('UseType1', '')).upper()
    emv_building = row.get('EMVBuilding1', 0)

    if 'VACANT' in desc or emv_building == 0:
        if 'RESIDENTIAL' in desc:
            return 'Vacant Residential Land'
        elif 'COMMERCIAL' in desc:
            return 'Vacant Commercial Land'
        elif 'INDUSTRIAL' in desc:
            return 'Vacant Industrial Land'
        else:
            return 'Vacant Other Land'

    if any(x in desc for x in [
        'SINGLE FAMILY', 'TWIN HOME', 'TOWNHOME', 'SINGLE FAMILY W/ACCESSORY'
    ]) or '1A RES' in use_type:
        return 'Single Family Residential'

    if any(x in desc for x in [
        'TWO FAMILY', 'THREE FAMILY', 'FOURPLEX',
        'APARTMENTS 4-6', '2ND RESID 4+ UNITS'
    ]):
        return 'Small Multi-Family (2-4 units)'

    if any(x in desc for x in [
        'APARTMENTS 7-19', 'APARTMENTS 20-49', 'APARTMENTS 50-99',
        'APT OR COMPLEX 100+', 'ASSISTED LIVING', 'NURSING HOME'
    ]):
        return 'Large Multi-Family (5+ units)'

    if 'CONDO' in desc:
        return 'Condominium'

    if 'MIXED' in desc:
        return 'Mixed-Use'

    if any(x in desc for x in [
        'RETAIL', 'STORE', 'SHOP', 'RESTAURANT',
        'HOTEL', 'MOTEL', 'OFFICE', 'BANK',
        'CLINIC', 'THEATER', 'SHOPPING CENTER',
        'SUPERMARKET', 'AUTO DEALER'
    ]):
        return 'Commercial'

    if any(x in desc for x in [
        'INDUSTRIAL', 'WAREHOUSE', 'MANUFACTURING', 'FACTORY',
        'FOUNDERY', 'R&D', 'TRUCK TERMINAL'
    ]):
        return 'Industrial'

    if 'PARKING' in desc:
        return 'Parking'

    if any(x in desc for x in [
        'CHURCH', 'EXEMPT', 'RAILROAD', 'FRATERNITY', 'GOLF COURSE'
    ]):
        return 'Institutional / Special Use'

    return 'Other'

st_paul_city['PROPERTY_CATEGORY'] = st_paul_city.apply(categorize_st_paul_property, axis=1)

# Consolidate vacant categories
vacant_mask = st_paul_city['LandUseCodeDescription'].str.contains('VACANT', case=False, na=False)
st_paul_city.loc[vacant_mask, 'PROPERTY_CATEGORY'] = 'Vacant Land'

print('Property Category Distribution:')
print(st_paul_city['PROPERTY_CATEGORY'].value_counts())

## Step 5: Calculate IR, Split Tax Capacity, Run Model

TC split-rate approach applied to the **full tax bill** (all jurisdictions).

In [ ]:
# Calculate Improvement Ratio and split Tax Capacity
st_paul_city['IR'] = st_paul_city['EMVBuilding1'] / st_paul_city['EMVTotal1']
st_paul_city['IR'] = st_paul_city['IR'].fillna(0)

st_paul_city['TaxCapacity_Improvements'] = st_paul_city['IR'] * st_paul_city['TaxCapacity']
st_paul_city['TaxCapacity_Land'] = (1 - st_paul_city['IR']) * st_paul_city['TaxCapacity']

print('Tax Capacity Split Summary:')
print(f'  Total Tax Capacity:          ${st_paul_city["TaxCapacity"].sum():>15,.0f}')
print(f'  Tax Capacity (Improvements): ${st_paul_city["TaxCapacity_Improvements"].sum():>15,.0f}')
print(f'  Tax Capacity (Land):         ${st_paul_city["TaxCapacity_Land"].sum():>15,.0f}')
print(f'  Land % of Tax Capacity:      {st_paul_city["TaxCapacity_Land"].sum() / st_paul_city["TaxCapacity"].sum() * 100:.1f}%')

# Run split-rate model at 4:1 ratio on FULL tax bill
tc_land_improvement_ratio = 4

tc_land_millage, tc_imp_millage, tc_split_rate_revenue, st_paul_city = model_split_rate_tax(
    df=st_paul_city,
    land_value_col='TaxCapacity_Land',
    improvement_value_col='TaxCapacity_Improvements',
    current_revenue=current_revenue,
    land_improvement_ratio=tc_land_improvement_ratio
)

st_paul_city['new_tax_tc'] = st_paul_city['new_tax']
st_paul_city['tax_change_tc'] = st_paul_city['new_tax_tc'] - st_paul_city['current_tax']
st_paul_city['tax_change_pct_tc'] = np.where(
    st_paul_city['current_tax'] > 0,
    (st_paul_city['tax_change_tc'] / st_paul_city['current_tax']) * 100,
    0
)

print(f'\nFull-Bill Split-Rate Model ({tc_land_improvement_ratio}:1 ratio)')
print(f'  Land Millage:        {tc_land_millage:.6f}')
print(f'  Improvement Millage: {tc_imp_millage:.6f}')
print(f'  Current Revenue:     ${current_revenue:,.0f}')
print(f'  New Revenue:         ${st_paul_city["new_tax_tc"].sum():,.0f}')
print(f'  Revenue neutral:     {abs(current_revenue - st_paul_city["new_tax_tc"].sum()) < 1}')

# Undeveloped and underdeveloped land analysis
vacant_results = analyze_vacant_land(
    st_paul_city,
    land_value_col='EMVLand1',
    property_type_col='PROPERTY_CATEGORY',
    vacant_identifier='Vacant Land',
    improvement_value_col='EMVBuilding1'
)
underdeveloped_results = analyze_land_by_improvement_share(
    st_paul_city,
    land_value_col='EMVLand1',
    improvement_value_col='EMVBuilding1'
)

total_land_emv = st_paul_city['EMVLand1'].sum()
print(f'\nUndeveloped and Underdeveloped Land')
print(f'  Total non-exempt land EMV: ${total_land_emv:,.0f}')
print(f'\n  Undeveloped (vacant, IR=0):')
print(f'    {vacant_results["total_vacant_parcels"]:,} parcels')
print(f'    ${vacant_results["total_vacant_land_value"]:,.0f} ({vacant_results["vacant_land_pct_of_total"]:.1f}% of non-exempt land value)')
print(f'\n  Underdeveloped (by improvement share):')
for cat in underdeveloped_results['categories']:
    print(f'    {cat["category"]:35s} {cat["parcel_count"]:>6,} parcels  ${cat["adjusted_land_value"]:>15,.0f}  ({cat["share_of_total_land_value_pct"]:.1f}%)')


## Step 6: Category Summary & Charts

In [ ]:
# Tax impact summary by property category
st_paul_city['abs_tax_diff'] = (st_paul_city['current_tax'] - st_paul_city['new_tax_tc']).abs()
total_abs_tax_diff = st_paul_city['abs_tax_diff'].sum()
percent_of_current = (total_abs_tax_diff / current_revenue) * 100
print(f'Sum of absolute tax difference: ${total_abs_tax_diff:,.2f}')
print(f'That is {percent_of_current:.2f}% of current full tax revenue.\n')

output_summary = calculate_category_tax_summary(
    st_paul_city,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax_tc'
)
print_category_tax_summary(output_summary, 'Full-Bill Split-Rate Impact by Property Category')

In [ ]:
# Horizontal bar chart: median % change by category
import matplotlib.pyplot as plt

filtered = output_summary[output_summary['property_count'] > 50].copy()

categories = filtered['PROPERTY_CATEGORY'].tolist()
counts = filtered['property_count'].tolist()
median_pct_change = filtered['median_tax_change_pct'].tolist()
median_dollar_change = filtered['median_tax_change'].tolist()
total_tax_change = filtered['total_tax_change'].tolist() if 'total_tax_change' in filtered.columns else (filtered['mean_tax_change'] * filtered['property_count']).tolist()

sorted_idx = np.argsort(median_pct_change)
categories = [categories[i] for i in sorted_idx]
counts = [counts[i] for i in sorted_idx]
median_pct_change = [median_pct_change[i] for i in sorted_idx]
median_dollar_change = [median_dollar_change[i] for i in sorted_idx]
total_tax_change = [total_tax_change[i] for i in sorted_idx]

bar_colors = ['#8B0000' if val > 0 else '#228B22' for val in median_pct_change]

bar_height = 0.75
fig_height = len(categories) * 0.8 + 1.2
right_col_pad = 120
fig, ax = plt.subplots(figsize=(17, fig_height))

y = np.arange(len(categories))

ax.barh(y, median_pct_change, color=bar_colors, edgecolor='none',
        height=bar_height, alpha=0.92, linewidth=0, zorder=2)

for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

cat_offset = 0.18
med_offset = -0.03
count_offset = -0.23

max_abs = max(abs(min(median_pct_change)), abs(max(median_pct_change)))
right_col_x = max_abs + right_col_pad

ax.text(right_col_x, len(categories) - 0.5, 'Net Change', va='bottom', ha='left',
        fontsize=15, fontweight='bold', color='black', fontname='Arial')

for i, (cat, val, count, med_dol, tot_change) in enumerate(
    zip(categories, median_pct_change, counts, median_dollar_change, total_tax_change)):
    med_dol_str = f'${med_dol:,.0f}' if med_dol >= 0 else f'-${abs(med_dol):,.0f}'
    pct_str = f'{val:+.1f}%'
    median_combo = f'Median: {med_dol_str}, {pct_str}'

    xpos = val - 2.5 if val < 0 else val + 2.5
    ha = 'right' if val < 0 else 'left'

    ax.text(xpos, y[i]+cat_offset, cat, va='center', ha=ha,
            fontsize=14, fontweight='bold', color='#222', fontname='Arial')
    ax.text(xpos, y[i]+med_offset, median_combo, va='center', ha=ha,
            fontsize=12, fontweight='bold', color='black', fontname='Arial')
    ax.text(xpos, y[i]+count_offset, f'{count:,} parcels', va='center', ha=ha,
            fontsize=11, fontweight='bold', color='#888', fontname='Arial')

    tot_change_str = f'${tot_change:,.0f}' if tot_change >= 0 else f'-${abs(tot_change):,.0f}'
    ax.text(right_col_x, y[i], tot_change_str, va='center', ha='left',
            fontsize=13, fontweight='bold', color='black', fontname='Arial')

ax.set_xlim(-right_col_x, right_col_x + 60)
ax.set_yticks([])
ax.set_xticks([])

plt.tight_layout()
plt.show()

In [ ]:
# Butterfly chart: percent of parcels increasing/decreasing >10%
import matplotlib.pyplot as plt

summary_filtered = output_summary[output_summary['property_count'] > 50].copy()
summary_sorted = summary_filtered.sort_values('pct_increase_gt_threshold', ascending=True)

categories_sorted = summary_sorted['PROPERTY_CATEGORY'].tolist()
pct_increase_sorted = summary_sorted['pct_increase_gt_threshold'].tolist()
pct_decrease_sorted = summary_sorted['pct_decrease_gt_threshold'].tolist()

pct_increase_int = [int(round(x)) for x in pct_increase_sorted]
pct_decrease_int = [int(round(x)) for x in pct_decrease_sorted]

y = np.arange(len(categories_sorted))
fig, ax = plt.subplots(figsize=(8, 6))

color_increase = '#8B0000'
color_decrease = '#228B22'

ax.barh(y, [-v for v in pct_decrease_sorted], color=color_decrease, edgecolor='none', height=0.7)
ax.barh(y, pct_increase_sorted, color=color_increase, edgecolor='none', height=0.7)

for i, (inc, dec) in enumerate(zip(pct_increase_int, pct_decrease_int)):
    if dec > 0:
        ax.text(-dec - 2, y[i], f'{dec}%', va='center', ha='right',
                fontsize=8, color=color_decrease, fontname='Arial')
    if inc > 0:
        ax.text(inc + 2, y[i], f'{inc}%', va='center', ha='left',
                fontsize=8, color=color_increase, fontname='Arial')

for i, (cat, inc) in enumerate(zip(categories_sorted, pct_increase_sorted)):
    xpos = inc + 18 if inc > 0 else 18
    ax.text(xpos, y[i], cat, va='center', ha='left',
            fontsize=9, fontweight='bold', color='#222', fontname='Arial')

for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
ax.set_yticks([])
ax.set_xticks([])

max_val = max(max(pct_increase_sorted), max(pct_decrease_sorted))
ax.set_xlim(-max_val-20, max_val+48)

title_y = len(categories_sorted) - 0.2
ax.text(-max_val * 0.45, title_y, 'Percent of parcels\ndecreasing >10%',
        ha='center', va='bottom', fontsize=10, color='black', fontname='Arial',
        bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.15'))
ax.text(max_val * 0.45, title_y, 'Percent of parcels\nincreasing >10%',
        ha='center', va='bottom', fontsize=10, color='black', fontname='Arial',
        bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.15'))

plt.tight_layout()
plt.show()

## Step 7: Scatter Plots

In [ ]:
# Scatter plot: Tax Change vs Improvement Ratio (full bill)
import matplotlib.pyplot as plt

# Use st_paul_city which already has the model results
plot_data = st_paul_city[
    (st_paul_city['current_tax'] > 100) & 
    (st_paul_city['IR'].notna()) &
    (st_paul_city['tax_change_pct_tc'].between(-100, 200))
].copy()

fig, ax = plt.subplots(figsize=(12, 8))

colors = np.where(plot_data['tax_change_pct_tc'] < 0, 'green', 'red')

ax.scatter(
    plot_data['IR'] * 100,
    plot_data['tax_change_pct_tc'],
    alpha=0.15, s=10, c=colors
)

ax.axhline(y=0, color='black', linestyle='-', linewidth=1)

system_avg_ir = (st_paul_city['EMVBuilding1'].sum() / st_paul_city['EMVTotal1'].sum()) * 100
ax.axvline(x=system_avg_ir, color='blue', linestyle='--', linewidth=1.5,
           label=f'System Avg IR ({system_avg_ir:.1f}%)')

z = np.polyfit(plot_data['IR'] * 100, plot_data['tax_change_pct_tc'], 1)
p = np.poly1d(z)
x_line = np.linspace(0, 100, 100)
ax.plot(x_line, p(x_line), 'orange', linewidth=2, label='Trend line')

ax.scatter([], [], c='green', alpha=0.5, s=50, label='Tax Decrease')
ax.scatter([], [], c='red', alpha=0.5, s=50, label='Tax Increase')

ax.set_xlabel('Improvement Ratio (%)', fontsize=12)
ax.set_ylabel('Tax Change (%)', fontsize=12)
ax.set_title(f'Full-Bill Tax Change vs Improvement Ratio ({tc_land_improvement_ratio}:1 TC Split-Rate)',
             fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 100)

plt.tight_layout()
plt.show()

print(f'\nPlot shows {len(plot_data):,} parcels')
print(f'  Green (tax decrease): {(plot_data["tax_change_pct_tc"] < 0).sum():,}')
print(f'  Red (tax increase): {(plot_data["tax_change_pct_tc"] >= 0).sum():,}')

In [ ]:
# Scatter plot: Tax Change by Neighborhood (full bill)
import matplotlib.pyplot as plt

# Spatial join to District Council neighborhoods
dc_dir = os.path.join('data', 'District_Councils_3386664414246726701')
if os.path.isdir(dc_dir):
    neighborhoods = gpd.read_file(dc_dir)
    nbhd_name_col = 'planning_1'
else:
    neighborhoods = get_feature_data_with_geometry(
        'Saint_Paul_District_Councils',
        'https://services5.arcgis.com/09i2I0fNSnHEF5Ef/arcgis/rest/services',
        layer_id=0, paginate=True
    )
    neighborhoods = ensure_geodataframe(neighborhoods)
    nbhd_name_col = 'planningdistrictname'
neighborhoods = neighborhoods[[nbhd_name_col, 'geometry']].rename(columns={nbhd_name_col: 'Neighborhood'})
neighborhoods = neighborhoods.to_crs(epsg=3857)

plot_source_3857 = st_paul_city.to_crs(epsg=3857)
plot_source_3857['centroid'] = plot_source_3857.geometry.centroid
centroids = gpd.GeoDataFrame(
    plot_source_3857.drop(columns='geometry'), geometry='centroid', crs='EPSG:3857'
)
plot_with_nbhd = gpd.sjoin(centroids, neighborhoods, how='left', predicate='within')
plot_with_nbhd = plot_with_nbhd.drop(columns=['index_right', 'centroid'])

plot_data = plot_with_nbhd[
    (plot_with_nbhd['current_tax'] > 100) & 
    (plot_with_nbhd['Neighborhood'].notna()) &
    (plot_with_nbhd['tax_change_pct_tc'].between(-100, 200))
].copy()

fig, ax = plt.subplots(figsize=(12, 8))

colors = np.where(plot_data['tax_change_pct_tc'] < 0, 'green', 'red')
ax.scatter(plot_data['IR'] * 100, plot_data['tax_change_pct_tc'],
           alpha=0.15, s=10, c=colors)

# Neighborhood medians
neighborhood_stats = plot_data.groupby('Neighborhood').agg({
    'IR': 'median',
    'tax_change_pct_tc': 'median',
    'TaxCapacity': 'count'
}).reset_index()
neighborhood_stats.columns = ['Neighborhood', 'median_IR', 'median_tax_change', 'count']
neighborhood_stats = neighborhood_stats.sort_values('median_tax_change', ascending=False)

cmap_colors = plt.cm.tab20(np.linspace(0, 1, len(neighborhood_stats)))
for i, row in enumerate(neighborhood_stats.itertuples()):
    ax.scatter(
        row.median_IR * 100, row.median_tax_change,
        s=200, c=[cmap_colors[i]], edgecolors='black', linewidths=1.5,
        label=f'{row.Neighborhood} ({row.median_tax_change:+.1f}%)', zorder=5
    )

ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
system_avg_ir = (st_paul_city['EMVBuilding1'].sum() / st_paul_city['EMVTotal1'].sum()) * 100
ax.axvline(x=system_avg_ir, color='blue', linestyle='--', linewidth=1.5)

ax.set_xlabel('Improvement Ratio (%)', fontsize=12)
ax.set_ylabel('Tax Change (%)', fontsize=12)
ax.set_title(f'Full-Bill Tax Change by Neighborhood ({tc_land_improvement_ratio}:1 TC Split-Rate)',
             fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=8, title='Neighborhood Medians')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 100)

plt.tight_layout()
plt.show()

print(f'\nPlot shows {len(plot_data):,} parcels across {plot_data["Neighborhood"].nunique()} neighborhoods')


In [ ]:
# Downtown tax change by property category
import matplotlib.pyplot as plt
import numpy as np

downtown = plot_with_nbhd[plot_with_nbhd['Neighborhood'] == 'Downtown'].copy()

dt_summary = downtown.groupby('PROPERTY_CATEGORY').agg(
    count=('tax_change_pct_tc', 'size'),
    median_change=('tax_change_pct_tc', 'median'),
    total_current=('current_tax', 'sum'),
    total_new=('new_tax_tc', 'sum')
).reset_index()
dt_summary['total_change'] = dt_summary['total_new'] - dt_summary['total_current']
dt_summary = dt_summary.sort_values('median_change')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left: median % change by category
colors = ['green' if x < 0 else 'red' for x in dt_summary['median_change']]
bars = ax1.barh(dt_summary['PROPERTY_CATEGORY'], dt_summary['median_change'], color=colors, alpha=0.7)
for bar, count in zip(bars, dt_summary['count']):
    x = bar.get_width()
    ax1.text(x + (1 if x >= 0 else -1), bar.get_y() + bar.get_height()/2,
             f'n={count}', va='center', fontsize=9)
ax1.axvline(x=0, color='black', linewidth=0.8)
ax1.set_xlabel('Median Tax Change (%)')
ax1.set_title('Downtown: Median Tax Change by Property Type')
ax1.grid(True, alpha=0.3, axis='x')

# Right: total dollar change by category
colors2 = ['green' if x < 0 else 'red' for x in dt_summary['total_change']]
ax2.barh(dt_summary['PROPERTY_CATEGORY'], dt_summary['total_change'] / 1000, color=colors2, alpha=0.7)
ax2.axvline(x=0, color='black', linewidth=0.8)
ax2.set_xlabel('Total Tax Change ($K)')
ax2.set_title('Downtown: Total Tax Change by Property Type')
ax2.grid(True, alpha=0.3, axis='x')

plt.suptitle(f'Downtown St. Paul — {len(downtown):,} parcels, ${downtown["current_tax"].sum()/1e6:.1f}M current tax',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nDowntown summary:")
for _, row in dt_summary.iterrows():
    print(f"  {row['PROPERTY_CATEGORY']:35s} n={row['count']:>4}  median: {row['median_change']:>+6.1f}%  total: ${row['total_change']:>+12,.0f}")

## Step 8: Census Equity Analysis

In [ ]:
# Get census data for Ramsey County (cached to disk; the API is fetched only once)
import concurrent.futures

census_cache = os.path.join(save_dir, 'ramsey_census_bg_2022.gpq')
census_boundaries = None
if os.path.exists(census_cache):
    census_boundaries = gpd.read_parquet(census_cache)
    if census_boundaries.crs is None:
        census_boundaries = census_boundaries.set_crs(epsg=4326)
    print(f'Loaded cached census boundaries: {len(census_boundaries)} block groups')
else:
    def _fetch_stpaul():
        return get_census_data_with_boundaries(fips_code='27123', year=2022)

    _ex = concurrent.futures.ThreadPoolExecutor(max_workers=1)
    _fut = _ex.submit(_fetch_stpaul)
    try:
        census_data, census_boundaries = _fut.result(timeout=90)
        census_boundaries = census_boundaries.set_crs(epsg=4326)
        census_boundaries.to_parquet(census_cache)
    except Exception as e:
        print(f'Census fetch failed ({type(e).__name__}: {e}); continuing without census')
        census_boundaries = None
    finally:
        _ex.shutdown(wait=False)

_census_cols = ['std_geoid', 'median_income', 'minority_pct', 'black_pct',
                'total_pop', 'white_pop', 'black_pop', 'hispanic_pop']
if census_boundaries is not None:
    boundary_gdf = st_paul_city.to_crs(epsg=4326)
    df = match_to_census_blockgroups(gdf=boundary_gdf, census_gdf=census_boundaries, join_type='left')
    for _col in _census_cols:
        if _col in df.columns:
            st_paul_city[_col] = df[_col]
    _n = st_paul_city['median_income'].notna().sum()
    print(f'Census block groups: {len(census_boundaries)}')
    print(f'Parcels with census data: {_n:,} of {len(st_paul_city):,} ({_n/len(st_paul_city)*100:.1f}%)')
else:
    # Define df so the downstream equity cells never NameError (they will simply be empty)
    for _col in _census_cols:
        if _col not in st_paul_city.columns:
            st_paul_city[_col] = float('nan')
    df = st_paul_city
    print('Proceeding without census; equity charts will be empty.')


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def filter_data(df):
    df_filtered = df[df['median_income'] > 0].copy()
    non_vacant_df = df[(df['PROPERTY_CATEGORY'] != 'Vacant Land') & (df['median_income'] > 0)].copy()
    return df_filtered, non_vacant_df

def calculate_block_group_summary(df):
    df = df[df['median_income'] > 0].copy()
    summary = df.groupby('std_geoid').agg(
        median_income=('median_income', 'first'),
        minority_pct=('minority_pct', 'first'),
        black_pct=('black_pct', 'first'),
        total_current_tax=('current_tax', 'sum'),
        total_new_tax=('new_tax_tc', 'sum'),
        mean_tax_change=('tax_change_tc', 'mean'),
        median_tax_change=('tax_change_tc', 'median'),
        median_tax_change_pct=('tax_change_pct_tc', 'median'),
        parcel_count=('tax_change_tc', 'count'),
        has_vacant_land=('PROPERTY_CATEGORY', lambda x: 'Vacant Land' in x.values)
    ).reset_index()
    summary = summary[summary['median_income'] > 0].copy()
    summary['mean_tax_change_pct'] = ((summary['total_new_tax'] - summary['total_current_tax']) / 
                                    summary['total_current_tax']) * 100
    return summary

def weighted_median(values, weights):
    mask = (~np.isnan(values)) & (~np.isnan(weights))
    values = np.array(values)[mask]
    weights = np.array(weights)[mask]
    if len(values) == 0:
        return np.nan
    sorter = np.argsort(values)
    values = values[sorter]
    weights = weights[sorter]
    cumsum = np.cumsum(weights)
    cutoff = weights.sum() / 2.0
    return values[np.searchsorted(cumsum, cutoff)]

def create_quintile_summary(df, group_col, value_col):
    if group_col == 'median_income':
        df = df[df['median_income'] > 0].copy()
    df[f'{group_col}_quintile'] = pd.qcut(df[group_col], 5,
                                         labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4', 'Q5 (Highest)'])
    summary = df.groupby(f'{group_col}_quintile').apply(
        lambda g: pd.Series({
            'count': g['tax_change_tc'].count(),
            'mean_tax_change_pct': g['tax_change_pct_tc'].mean(),
            'median_tax_change_pct': weighted_median(g['tax_change_pct_tc'].values, np.ones(len(g))),
            'mean_value': g[value_col].mean()
        })
    ).reset_index()
    return summary

# Run analysis
gdf_filtered, non_vacant_gdf = filter_data(df)
print(f'Filtered parcels: {len(gdf_filtered):,}')
print(f'Non-vacant parcels: {len(non_vacant_gdf):,}')

income_quintile_summary = create_quintile_summary(gdf_filtered, 'median_income', 'median_income')
non_vacant_income_quintile_summary = create_quintile_summary(non_vacant_gdf, 'median_income', 'median_income')
minority_quintile_summary = create_quintile_summary(gdf_filtered, 'minority_pct', 'minority_pct')
non_vacant_minority_quintile_summary = create_quintile_summary(non_vacant_gdf, 'minority_pct', 'minority_pct')

print('\nTax impact by income quintile (all properties):')
display(income_quintile_summary)
print('\nTax impact by income quintile (excluding vacant land):')
display(non_vacant_income_quintile_summary)
print('\nTax impact by minority % quintile (all properties):')
display(minority_quintile_summary)
print('\nTax impact by minority % quintile (excluding vacant land):')
display(non_vacant_minority_quintile_summary)

In [ ]:
# Quintile line plots
plt.figure(figsize=(10, 6))
plt.plot(income_quintile_summary['median_income_quintile'],
         income_quintile_summary['mean_tax_change_pct'], marker='o', label='All Properties')
plt.plot(non_vacant_income_quintile_summary['median_income_quintile'],
         non_vacant_income_quintile_summary['mean_tax_change_pct'], marker='o', label='Excluding Vacant Land')
plt.xlabel('Median Income Quintile')
plt.ylabel('Mean Tax Change (%)')
plt.title('Full-Bill: Mean Tax Change by Median Income Quintile')
plt.legend()
plt.axhline(0, color='black', linewidth=1, linestyle='dotted')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(minority_quintile_summary['minority_pct_quintile'],
         minority_quintile_summary['mean_tax_change_pct'], marker='o', label='All Properties')
plt.plot(non_vacant_minority_quintile_summary['minority_pct_quintile'],
         non_vacant_minority_quintile_summary['mean_tax_change_pct'], marker='o', label='Excluding Vacant Land')
plt.xlabel('Minority Percentage Quintile')
plt.ylabel('Mean Tax Change (%)')
plt.title('Full-Bill: Mean Tax Change by Minority Percentage Quintile')
plt.legend()
plt.axhline(0, color='black', linewidth=1, linestyle='dotted')
plt.tight_layout()
plt.show()

In [ ]:
# Inverted bar charts: median tax change by quintile (excluding vacant land)
sns.set_theme(style='whitegrid', font_scale=1.15)

# Income quintiles
fig, ax = plt.subplots(figsize=(10, 6))
vals = non_vacant_income_quintile_summary['median_tax_change_pct']
labels = non_vacant_income_quintile_summary['median_income_quintile']
colors = sns.color_palette('Greens', n_colors=len(vals))
color_map = [colors[i] for i in np.argsort(np.argsort(-vals))]

bars = ax.bar(labels, vals, color=color_map, edgecolor='black', width=0.7)
ax.yaxis.set_visible(False)
ax.set_title('Median Tax Change by Neighborhood Income (Excl. Vacant)', weight='bold', pad=30)
sns.despine(left=True, right=True, top=True, bottom=True)
for bar, val in zip(bars, vals):
    ax.annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, val / 2),
                ha='center', va='center', fontsize=13, fontweight='bold')
ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')
plt.xticks(fontweight='bold')
margin = max(abs(vals.min()), abs(vals.max())) * 1.2
ax.set_ylim(-margin, margin)
ax.axhline(y=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

# Minority quintiles
fig, ax = plt.subplots(figsize=(10, 6))
vals2 = non_vacant_minority_quintile_summary['median_tax_change_pct']
labels2 = non_vacant_minority_quintile_summary['minority_pct_quintile']
colors2 = sns.color_palette('Greens', n_colors=len(vals2))
color_map2 = [colors2[i] for i in np.argsort(np.argsort(-vals2))]

bars2 = ax.bar(labels2, vals2, color=color_map2, edgecolor='black', width=0.7)
ax.yaxis.set_visible(False)
ax.set_title('Median Tax Change by Minority % Quintile (Excl. Vacant)', weight='bold', pad=30)
sns.despine(left=True, right=True, top=True, bottom=True)
for bar, val in zip(bars2, vals2):
    ax.annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, val / 2),
                ha='center', va='center', fontsize=13, fontweight='bold')
ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')
plt.xticks(fontweight='bold')
margin2 = max(abs(vals2.min()), abs(vals2.max())) * 1.2
ax.set_ylim(-margin2, margin2)
ax.axhline(y=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Inverted bar charts: single family residential only
sfr = df[(df['PROPERTY_CATEGORY'] == 'Single Family Residential') & (df['median_income'] > 0)].copy()
print(f'Single family parcels: {len(sfr):,}')

sfr_income_quintile = create_quintile_summary(sfr, 'median_income', 'median_income')
sfr_minority_quintile = create_quintile_summary(sfr, 'minority_pct', 'minority_pct')

sns.set_theme(style='whitegrid', font_scale=1.15)

# Income quintiles - single family
fig, ax = plt.subplots(figsize=(10, 6))
vals = sfr_income_quintile['median_tax_change_pct']
labels = sfr_income_quintile['median_income_quintile']
colors = sns.color_palette('Greens', n_colors=len(vals))
color_map = [colors[i] for i in np.argsort(np.argsort(-vals))]
bars = ax.bar(labels, vals, color=color_map, edgecolor='black', width=0.7)
ax.yaxis.set_visible(False)
ax.set_title('Median Tax Change by Neighborhood Income (Single Family Only)', weight='bold', pad=30)
sns.despine(left=True, right=True, top=True, bottom=True)
for bar, val in zip(bars, vals):
    ax.annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, val / 2),
                ha='center', va='center', fontsize=13, fontweight='bold')
ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')
plt.xticks(fontweight='bold')
margin = max(abs(vals.min()), abs(vals.max())) * 1.2
ax.set_ylim(-margin, margin)
ax.axhline(y=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

# Minority quintiles - single family
fig, ax = plt.subplots(figsize=(10, 6))
vals2 = sfr_minority_quintile['median_tax_change_pct']
labels2 = sfr_minority_quintile['minority_pct_quintile']
colors2 = sns.color_palette('Greens', n_colors=len(vals2))
color_map2 = [colors2[i] for i in np.argsort(np.argsort(-vals2))]
bars2 = ax.bar(labels2, vals2, color=color_map2, edgecolor='black', width=0.7)
ax.yaxis.set_visible(False)
ax.set_title('Full-Bill: Median Tax Change by Minority % Quintile (Single Family Only)', weight='bold', pad=30)
sns.despine(left=True, right=True, top=True, bottom=True)
for bar, val in zip(bars2, vals2):
    ax.annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, val / 2),
                ha='center', va='center', fontsize=13, fontweight='bold')
ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')
plt.xticks(fontweight='bold')
margin2 = max(abs(vals2.min()), abs(vals2.max())) * 1.2
ax.set_ylim(-margin2, margin2)
ax.axhline(y=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Inverted bar charts: multifamily residential only
mfr = df[(df['PROPERTY_CATEGORY'].isin(['Small Multi-Family (2-4 units)', 'Large Multi-Family (5+ units)'])) & (df['median_income'] > 0)].copy()
print(f'Multifamily parcels: {len(mfr):,}')

mfr_income_quintile = create_quintile_summary(mfr, 'median_income', 'median_income')
mfr_minority_quintile = create_quintile_summary(mfr, 'minority_pct', 'minority_pct')

sns.set_theme(style='whitegrid', font_scale=1.15)

# Income quintiles - multifamily
fig, ax = plt.subplots(figsize=(10, 6))
vals = mfr_income_quintile['median_tax_change_pct']
labels = mfr_income_quintile['median_income_quintile']
colors = sns.color_palette('Greens', n_colors=len(vals))
color_map = [colors[i] for i in np.argsort(np.argsort(-vals))]
bars = ax.bar(labels, vals, color=color_map, edgecolor='black', width=0.7)
ax.yaxis.set_visible(False)
ax.set_title('Full-Bill: Median Tax Change by Neighborhood Income (Multifamily Only)', weight='bold', pad=30)
sns.despine(left=True, right=True, top=True, bottom=True)
for bar, val in zip(bars, vals):
    ax.annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, val / 2),
                ha='center', va='center', fontsize=13, fontweight='bold')
ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')
plt.xticks(fontweight='bold')
margin = max(abs(vals.min()), abs(vals.max())) * 1.2
ax.set_ylim(-margin, margin)
ax.axhline(y=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

# Minority quintiles - multifamily
fig, ax = plt.subplots(figsize=(10, 6))
vals2 = mfr_minority_quintile['median_tax_change_pct']
labels2 = mfr_minority_quintile['minority_pct_quintile']
colors2 = sns.color_palette('Greens', n_colors=len(vals2))
color_map2 = [colors2[i] for i in np.argsort(np.argsort(-vals2))]
bars2 = ax.bar(labels2, vals2, color=color_map2, edgecolor='black', width=0.7)
ax.yaxis.set_visible(False)
ax.set_title('Full-Bill: Median Tax Change by Minority % Quintile (Multifamily Only)', weight='bold', pad=30)
sns.despine(left=True, right=True, top=True, bottom=True)
for bar, val in zip(bars2, vals2):
    ax.annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, val / 2),
                ha='center', va='center', fontsize=13, fontweight='bold')
ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')
plt.xticks(fontweight='bold')
margin2 = max(abs(vals2.min()), abs(vals2.max())) * 1.2
ax.set_ylim(-margin2, margin2)
ax.axhline(y=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Inverted bar charts: all residential (SFR + multifamily)
smfr = df[(df['PROPERTY_CATEGORY'].isin(['Single Family Residential','Small Multi-Family (2-4 units)', 'Large Multi-Family (5+ units)'])) & (df['median_income'] > 0)].copy()
print(f'Residential parcels: {len(smfr):,}')

smfr_income_quintile = create_quintile_summary(smfr, 'median_income', 'median_income')
smfr_minority_quintile = create_quintile_summary(smfr, 'minority_pct', 'minority_pct')

sns.set_theme(style='whitegrid', font_scale=1.15)

# Income quintiles - all residential
fig, ax = plt.subplots(figsize=(10, 6))
vals = smfr_income_quintile['median_tax_change_pct']
labels = smfr_income_quintile['median_income_quintile']
colors = sns.color_palette('Greens', n_colors=len(vals))
color_map = [colors[i] for i in np.argsort(np.argsort(-vals))]
bars = ax.bar(labels, vals, color=color_map, edgecolor='black', width=0.7)
ax.yaxis.set_visible(False)
ax.set_title('Full-Bill: Median Tax Change by Neighborhood Income (All Residential)', weight='bold', pad=30)
sns.despine(left=True, right=True, top=True, bottom=True)
for bar, val in zip(bars, vals):
    ax.annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, val / 2),
                ha='center', va='center', fontsize=13, fontweight='bold')
ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')
plt.xticks(fontweight='bold')
margin = max(abs(vals.min()), abs(vals.max())) * 1.2
ax.set_ylim(-margin, margin)
ax.axhline(y=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

# Minority quintiles - all residential
fig, ax = plt.subplots(figsize=(10, 6))
vals2 = smfr_minority_quintile['median_tax_change_pct']
labels2 = smfr_minority_quintile['minority_pct_quintile']
colors2 = sns.color_palette('Greens', n_colors=len(vals2))
color_map2 = [colors2[i] for i in np.argsort(np.argsort(-vals2))]
bars2 = ax.bar(labels2, vals2, color=color_map2, edgecolor='black', width=0.7)
ax.yaxis.set_visible(False)
ax.set_title('Full-Bill: Median Tax Change by Minority % Quintile (All Residential)', weight='bold', pad=30)
sns.despine(left=True, right=True, top=True, bottom=True)
for bar, val in zip(bars2, vals2):
    ax.annotate(f'{val:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, val / 2),
                ha='center', va='center', fontsize=13, fontweight='bold')
ax.xaxis.set_ticks_position('top')
ax.xaxis.set_label_position('top')
plt.xticks(fontweight='bold')
margin2 = max(abs(vals2.min()), abs(vals2.max())) * 1.2
ax.set_ylim(-margin2, margin2)
ax.axhline(y=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Save modeled data for use in st_paul_policy_analysis.ipynb
st_paul_city.to_parquet(os.path.join(save_dir, 'st_paul_city_modeled.gpq'))
st_paul_gdf.to_parquet(os.path.join(save_dir, 'st_paul_gdf.gpq'))
print(f'Saved {len(st_paul_city):,} city-taxable parcels and {len(st_paul_gdf):,} total parcels')


In [ ]:
# Export standardized CSV — do not remove or move above Census join
import sys
sys.path.insert(0, '../..')
from lvt.lvt_utils import save_standard_export

out_df = save_standard_export(
    df=st_paul_city,
    city='st_paul',
    output_path='../../analysis/data/st_paul.csv',
    model_type='split_rate:4.0',
    land_millage=tc_land_millage,
    improvement_millage=tc_imp_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax_tc',
    tax_change_col='tax_change_tc',
    tax_change_pct_col='tax_change_pct_tc',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)


In [ ]:
# Standard report: category impact, income quintile, distribution
import sys
sys.path.insert(0, '../..')
from lvt.viz import create_city_report

create_city_report(out_df, 'st_paul', show=True)


In [ ]:
# Interactive parcel map — GeoParquet + self-contained tax-change HTML
import sys
sys.path.insert(0, '../..')
import geopandas as _gpd
from lvt.parcel_map import save_parcel_map_export, create_parcel_map

# Reuse the exact modeled frame + model_type / millages / tax columns as the
# standard-export CSV so the map matches the CSV. Geometry is already on
# st_paul_city (unioned during the condo collapse; index reset by the concat).
_map_gdf = st_paul_city

_map_out = save_parcel_map_export(
    gdf=_map_gdf,
    city='st_paul',
    output_path='../../analysis/maps/st_paul.parquet',
    model_type='split_rate:4.0',
    land_millage=tc_land_millage,
    improvement_millage=tc_imp_millage,
    parcel_id_col='ParcelID',
    parcel_url_template=None,
    owner_name_col='OwnerName',
    owner_address_col='SiteAddress',
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax_tc',
    tax_change_col='tax_change_tc',
    tax_change_pct_col='tax_change_pct_tc',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

create_parcel_map(_map_out, 'st_paul', simplify_tolerance_m=2)
print('st_paul parcel map done.')

## Step 9: Transit Walk Sheds & Parking

How much land within a 10-minute walk of high-frequency transit (METRO light rail + BRT) is consumed by parking lots? Stops and route alignments come from the Metro Transit GTFS feed (Mobility Database mdb-205); parking lot footprints from OpenStreetMap. Walk sheds are network-routed isochrones — 800 m walked along the OSM street network from each stop — so rivers, rail yards, and freeway severance are respected. Parking near transit is high-value land held at low improvement ratios: exactly the parcels a land value tax pressures toward development.


In [ ]:
# Pull GTFS data from the Mobility Database and subset to METRO BRT / light rail routes
from lvt.transit_utils import download_gtfs_from_mobility_database, gtfs_route_stops

gtfs_path = download_gtfs_from_mobility_database(
    os.path.join(save_dir, 'gtfs_metro_transit.zip'),
    provider='Metro Transit', subdivision='Minnesota'
)

# METRO brand = light rail (route_type 0: Blue/Green) + BRT (route_type 3: Gold/Orange/Red + A-E arterial lines)
gtfs_data = gtfs_route_stops(gtfs_path, route_selector='METRO')
metro_routes = gtfs_data['routes']
stop_route_pairs = gtfs_data['stop_route_pairs']

# Keep stops inside St. Paul (District Council boundary from Step 7)
city_boundary_3857 = neighborhoods.union_all()
metro_stops_3857 = gtfs_data['stops'].to_crs(epsg=3857)
sp_stops = metro_stops_3857[metro_stops_3857.within(city_boundary_3857)].copy()

route_summary = (
    stop_route_pairs[stop_route_pairs['stop_id'].isin(set(sp_stops['stop_id']))]
    .groupby('route_id')['stop_id'].nunique()
    .rename('stops_in_st_paul').reset_index()
    .merge(metro_routes[['route_id', 'route_long_name', 'route_type', 'route_color']], on='route_id')
)

# Alignments only for the lines that actually serve St. Paul
route_lines = gtfs_data['route_lines'][gtfs_data['route_lines']['route_id'].isin(route_summary['route_id'])]

print(f'{len(sp_stops)} METRO BRT/LRT stops in St. Paul')
print(route_summary[['route_long_name', 'route_type', 'stops_in_st_paul']].to_string(index=False))


In [ ]:
# Routed 10-minute walk sheds per stop; parking lot vs taxable land and value, by line
from lvt.transit_utils import (
    get_walk_network, route_walk_sheds, fetch_osm_parking,
    flag_parking_parcels, walk_shed_stats, ACRE_SQM
)

WALK_M = 800  # 10 minutes at ~80 m/min, walked along the street network

sp_stops_utm = sp_stops.to_crs(epsg=26915)

# Route the sheds (cached: the walk graph and the resulting shed polygons)
sheds_path = os.path.join(save_dir, 'walk_sheds_800m.gpkg')
if os.path.exists(sheds_path):
    walk_sheds = gpd.read_file(sheds_path).to_crs(epsg=26915)
else:
    G_walk = get_walk_network(
        boundary_geom=neighborhoods.to_crs(epsg=26915).union_all(),
        boundary_crs='EPSG:26915',
        graph_path=os.path.join(save_dir, 'stpaul_walk.graphml')
    )
    walk_sheds = route_walk_sheds(G_walk, sp_stops_utm, id_col='stop_id', cutoff_m=WALK_M)
    walk_sheds.to_file(sheds_path)

# Clip sheds to the St. Paul boundary so all stats and the map exclude land in
# neighboring cities (the routed network extends ~1 km past the city line)
city_boundary_utm = neighborhoods.to_crs(epsg=26915).union_all()
walk_sheds['geometry'] = walk_sheds.geometry.intersection(city_boundary_utm).buffer(0)
walk_sheds = walk_sheds[~walk_sheds.geometry.is_empty]

shed_union = walk_sheds.union_all().buffer(0)

# OSM parking lots (cached); excludes underground garages and on-street lanes
osm_parking = fetch_osm_parking(
    neighborhoods, os.path.join(save_dir, 'osm_parking_st_paul.gpkg'), to_crs='EPSG:26915'
)
parking_union = osm_parking.union_all().buffer(0)

# Parcel layers prepared once: all parcels for land composition, city-taxable for value shares
parcels_utm = st_paul_gdf.to_crs(epsg=26915)
parcels_utm['geometry'] = parcels_utm['geometry'].buffer(0)
city_parcels_utm = flag_parking_parcels(
    st_paul_city.to_crs(epsg=26915), parking_union, category_col='PROPERTY_CATEGORY'
)

# Each line gets its own shed union and independent stats (sheds of lines sharing stops overlap)
line_rows = [walk_shed_stats(shed_union, parcels_utm, city_parcels_utm, parking_union,
                             label='All METRO', n_stops=len(sp_stops_utm))]
for _, rt in route_summary.iterrows():
    line_stop_ids = set(stop_route_pairs.loc[stop_route_pairs['route_id'] == rt['route_id'], 'stop_id'])
    line_sheds = walk_sheds[walk_sheds['stop_id'].isin(line_stop_ids)]
    line_rows.append(walk_shed_stats(line_sheds.union_all(), parcels_utm, city_parcels_utm, parking_union,
                                     label=rt['route_long_name'], n_stops=len(line_sheds)))

walk_shed_summary = pd.DataFrame(line_rows).set_index('line')

overall = walk_shed_summary.loc['All METRO']
print(f"Walk sheds: {overall['shed_acres']:,.0f} acres routed within {WALK_M} m of {len(sp_stops_utm)} stops")
print(f"Parking lots cover {overall['parking_pct_of_shed']:.1f}% of shed land and "
      f"{overall['parking_pct_of_taxable_land']:.1f}% of taxable parcel land")
print(f"Parking parcels hold {overall['parking_pct_of_parcel_land']:.1f}% of taxable parcel land and "
      f"{overall['parking_pct_of_land_value']:.1f}% of land value, but only "
      f"{overall['parking_pct_of_total_emv']:.1f}% of total EMV")
print(f"Full-bill LVT tax change: parking {overall['parking_tax_change_pct']:+.1f}% "
      f"vs non-parking {overall['nonparking_tax_change_pct']:+.1f}%")
display(walk_shed_summary.round(1))


In [ ]:
# Interactive map for export: routed walk sheds, parking by land value, transit lines by mode
import folium
import shapely
import branca.colormap as bcm
from folium import MacroElement
from jinja2 import Template as _JinjaTemplate


class _ZoomFade(MacroElement):
    """Fade a layer out as the map zooms in (light context when zoomed out, gone when zoomed in)."""

    def __init__(self, layer, fade_start=12.0, fade_end=14.0, max_fill=0.16, max_line=0.6):
        super().__init__()
        self._layer = layer
        self.fade_start = fade_start
        self.fade_end = fade_end
        self.max_fill = max_fill
        self.max_line = max_line
        self._template = _JinjaTemplate("""
            {% macro script(this, kwargs) %}
            (function(){
              var _map = {{ this._parent.get_name() }};
              var _lyr = {{ this._layer.get_name() }};
              function _fade(){
                var z = _map.getZoom();
                var t = Math.max(0, Math.min(1,
                    ({{ this.fade_end }} - z) / ({{ this.fade_end }} - {{ this.fade_start }})));
                _lyr.setStyle({fillOpacity: {{ this.max_fill }} * t, opacity: {{ this.max_line }} * t});
                var _lbls = document.getElementsByClassName('nbhd-label');
                for (var _i = 0; _i < _lbls.length; _i++) { _lbls[_i].style.opacity = t; }
              }
              _map.on('zoomend', _fade);
              _fade();
            })();
            {% endmacro %}
        """)

SQFT_PER_SQM = 10.7639
LV_VMAX_K = 2000  # cap the land-value color scale at $2M/acre for a clean, readable gradient
YLORRD = ['#ffffcc', '#ffeda0', '#fed976', '#feb24c', '#fd8d3c',
          '#fc4e2a', '#e31a1c', '#bd0026', '#800026']

# Attach parcel attributes to each parking lot in the sheds
parking_in_sheds = osm_parking[osm_parking.intersects(shed_union)].copy()
lot_points = parking_in_sheds[['osm_id', 'geometry']].copy()
lot_points['geometry'] = lot_points.representative_point()
taxable_attrs = city_parcels_utm[['PROPERTY_CATEGORY', 'parcel_sqm', 'parking_overlap_sqm',
                                  'EMVLand1', 'geometry']].copy()
taxable_attrs['land_value_per_acre'] = taxable_attrs['EMVLand1'] / (taxable_attrs['parcel_sqm'] / ACRE_SQM)
tax_join = gpd.sjoin(lot_points, taxable_attrs, how='left', predicate='within').drop_duplicates('osm_id')
parking_in_sheds = parking_in_sheds.merge(tax_join.drop(columns=['geometry', 'index_right']),
                                          on='osm_id', how='left')
# Lots not on a taxable parcel sit on tax-exempt land (colleges, hospitals, churches, city);
# pull owner and land use from the full parcel set for their tooltips
is_exempt = parking_in_sheds['land_value_per_acre'].isna()
exempt_pts = lot_points[lot_points['osm_id'].isin(parking_in_sheds.loc[is_exempt, 'osm_id'])]
exempt_join = gpd.sjoin(exempt_pts, parcels_utm[['OwnerName', 'LandUseCodeDescription', 'geometry']],
                        how='left', predicate='within').drop_duplicates('osm_id')
parking_in_sheds = parking_in_sheds.merge(
    exempt_join[['osm_id', 'OwnerName', 'LandUseCodeDescription']], on='osm_id', how='left')

paved_share = (parking_in_sheds['parking_overlap_sqm'] / parking_in_sheds['parcel_sqm']).clip(upper=1)
parking_in_sheds['land_value_k_per_acre'] = parking_in_sheds['land_value_per_acre'] / 1000
parking_in_sheds['parking_type'] = parking_in_sheds['parking_type'].replace('', 'unspecified')
parking_in_sheds['Parking type'] = parking_in_sheds['parking_type']
parking_in_sheds['Parcel category'] = parking_in_sheds['PROPERTY_CATEGORY']
parking_in_sheds['Total parcel land value'] = parking_in_sheds['EMVLand1'].map(
    lambda v: f'${v:,.0f}' if pd.notna(v) else '')
parking_in_sheds['Land value / acre'] = parking_in_sheds['land_value_per_acre'].map(
    lambda v: f'${v:,.0f}' if pd.notna(v) else '')
parking_in_sheds['Paved area (sq ft)'] = (parking_in_sheds['parking_overlap_sqm'] * SQFT_PER_SQM).map(
    lambda v: f'{v:,.0f}' if pd.notna(v) else '')
parking_in_sheds['Land value under parking'] = (parking_in_sheds['EMVLand1'] * paved_share).map(
    lambda v: f'${v:,.0f}' if pd.notna(v) else '')
parking_in_sheds['Owner'] = parking_in_sheds['OwnerName'].astype(str).str.title().replace('Nan', '—')
parking_in_sheds['Land use'] = parking_in_sheds['LandUseCodeDescription'].astype(str).str.title().replace('Nan', '—')
TAXED_TOOLTIP = ['Parking type', 'Parcel category', 'Total parcel land value', 'Land value / acre',
                 'Paved area (sq ft)', 'Land value under parking']
EXEMPT_TOOLTIP = ['Parking type', 'Owner', 'Land use']


def compact(gdf, tol_m=1.0):
    """Shrink a layer for inline embedding: simplify in meters, snap to ~1 m grid in degrees."""
    out = gdf.copy()
    if tol_m:
        out['geometry'] = out.geometry.simplify(tol_m)
    out = out.to_crs(epsg=4326)
    out['geometry'] = shapely.set_precision(out.geometry.values, 1e-5)
    return out


# Transit lines colored by mode; arterial BRT lines get distinct colors
ROUTE_STYLE = {
    'METRO Green Line':  {'color': '#008144', 'group': 'Light rail (LRT)'},
    'METRO Blue Line':   {'color': '#0053A0', 'group': 'Light rail (LRT)'},
    'METRO Gold Line':   {'color': "#F5BD2E", 'group': 'Bus rapid transit (BRT)'},
    'METRO Orange Line': {'color': '#F68B1F', 'group': 'Bus rapid transit (BRT)'},
    'METRO Red Line':    {'color': '#E71324', 'group': 'Bus rapid transit (BRT)'},
    'METRO A Line': {'color': '#4A1486', 'group': 'Arterial BRT (aBRT)'},
    'METRO B Line': {'color': '#7B3294', 'group': 'Arterial BRT (aBRT)'},
    'METRO C Line': {'color': '#A16BB1', 'group': 'Arterial BRT (aBRT)'},
    'METRO D Line': {'color': '#6A3D9A', 'group': 'Arterial BRT (aBRT)'},
    'METRO E Line': {'color': '#C994C7', 'group': 'Arterial BRT (aBRT)'},
}
DEFAULT_ROUTE = {'color': '#555555', 'group': 'Other'}

# Base map with the tile layer hidden from the layer control (no basemap URL in exports)
MAP_KWDS = dict(zoomSnap=0.25, zoomDelta=0.5, wheelPxPerZoomLevel=750, wheelDebounceTime=100)
city_latlon = gpd.GeoDataFrame(geometry=[city_boundary_utm], crs='EPSG:26915').to_crs(epsg=4326)
cminx, cminy, cmaxx, cmaxy = city_latlon.total_bounds
m = folium.Map(location=[(cminy + cmaxy) / 2, (cminx + cmaxx) / 2], tiles=None,
               zoom_start=12, control_scale=True, **MAP_KWDS)
folium.TileLayer('cartodbpositron', control=False).add_to(m)

# District Council neighborhoods: faint areas + name labels that fade out as you zoom in
neighborhoods_layer = folium.FeatureGroup(name='Neighborhoods (fade on zoom-in)')
folium.GeoJson(
    compact(neighborhoods.to_crs(epsg=26915)[['Neighborhood', 'geometry']], tol_m=8),
    style_function=lambda f: {'color': '#7a7a7a', 'weight': 0.8,
                              'fillColor': '#9aa0a6', 'fillOpacity': 0.16},
    tooltip=folium.GeoJsonTooltip(fields=['Neighborhood']),
).add_to(neighborhoods_layer)
_nbhd_utm = neighborhoods.to_crs(epsg=26915)
# Most labels sit at the district centroid; a couple need a manual nudge (east, north meters)
_LABEL_NUDGE_M = {
    'Saint Anthony Park': (-350, 600),     # move north + west (clear the border notch)
    'Macalester - Groveland': (-700, 0),   # move west
    'Payne - Phalen': (800, 0),            # move east toward Lake Phalen
    'Battle Creek - Conway - Eastview - Highwood Hills': (-700, 0),  # pull west off the east border
}
_label_points = []
for _, _row in _nbhd_utm.iterrows():
    _c = _row.geometry.centroid
    _dx, _dy = _LABEL_NUDGE_M.get(str(_row['Neighborhood']), (0.0, 0.0))
    _label_points.append(shapely.Point(_c.x + _dx, _c.y + _dy))
_nbhd_centroids = gpd.GeoSeries(_label_points, crs='EPSG:26915').to_crs(epsg=4326)
for (_, _row), _pt in zip(_nbhd_utm.iterrows(), _nbhd_centroids):
    _name = str(_row['Neighborhood'])
    _segs = _name.split(' - ')
    if len(_name) > 28 and len(_segs) >= 3:
        _half = (len(_segs) + 1) // 2
        _html = ' - '.join(_segs[:_half]) + ' -<br>' + ' - '.join(_segs[_half:])
        _anchor = (85, 17)
    else:
        _html = _name
        _anchor = (85, 9)
    folium.Marker(
        [_pt.y, _pt.x],
        icon=folium.DivIcon(icon_size=(170, 34), icon_anchor=_anchor,
                            html='<div class="nbhd-label">' + _html + '</div>'),
    ).add_to(neighborhoods_layer)
neighborhoods_layer.add_to(m)
m.get_root().header.add_child(folium.Element(
    '<style>.leaflet-div-icon{background:transparent;border:none;}'
    '.nbhd-label{font:600 11px Helvetica,Arial,sans-serif;color:#5f5f5f;text-align:center;'
    'white-space:nowrap;pointer-events:none;'
    'text-shadow:0 0 2px #fff,0 0 3px #fff,0 0 3px #fff;}</style>'))

parking_cmap = bcm.LinearColormap(YLORRD, vmin=0, vmax=LV_VMAX_K)

folium.GeoJson(
    compact(gpd.GeoDataFrame(geometry=[shed_union], crs='EPSG:26915'), tol_m=3), name='10-min walk shed',
    style_function=lambda f: {'color': '#5b7fa6', 'weight': 1.5, 'fillColor': '#5b7fa6', 'fillOpacity': 0.05}
).add_to(m)
folium.GeoJson(
    compact(gpd.GeoDataFrame(geometry=[city_boundary_utm], crs='EPSG:26915'), tol_m=3), name='St. Paul boundary',
    style_function=lambda f: {'color': '#333333', 'weight': 2.5, 'fill': False, 'dashArray': '6 3'}
).add_to(m)

exempt = parking_in_sheds[is_exempt.values]
folium.GeoJson(
    compact(exempt[EXEMPT_TOOLTIP + ['geometry']], tol_m=1.5), name='Parking — tax-exempt land',
    style_function=lambda f: {'color': '#9a9a9a', 'weight': 0.4, 'fillColor': '#c0c0c0', 'fillOpacity': 0.55},
    tooltip=folium.GeoJsonTooltip(fields=EXEMPT_TOOLTIP)
).add_to(m)
valued = parking_in_sheds[~is_exempt.values]
folium.GeoJson(
    compact(valued[TAXED_TOOLTIP + ['land_value_k_per_acre', 'geometry']], tol_m=1.5),
    name='Parking — taxed (by land value)',
    style_function=lambda f: {'color': '#8b0000', 'weight': 0.4,
        'fillColor': parking_cmap(min(f['properties']['land_value_k_per_acre'] or 0, LV_VMAX_K)),
        'fillOpacity': 0.85},
    tooltip=folium.GeoJsonTooltip(fields=TAXED_TOOLTIP)
).add_to(m)

# Full route alignments (not clipped to the city), grouped into one toggle
route_group = folium.FeatureGroup(name='Transit routes').add_to(m)
for _, rt in route_lines.to_crs(epsg=4326).iterrows():
    color = ROUTE_STYLE.get(rt['route_name'], DEFAULT_ROUTE)['color']
    folium.GeoJson(
        compact(gpd.GeoDataFrame([rt.drop('geometry')], geometry=[rt.geometry], crs='EPSG:4326'), tol_m=5),
        style_function=lambda f, c=color: {'color': c, 'weight': 4, 'opacity': 0.95}
    ).add_to(route_group)
stop_group = folium.FeatureGroup(name='METRO stops').add_to(m)
for _, s in sp_stops_utm.to_crs(epsg=4326).iterrows():
    folium.CircleMarker([s.geometry.y, s.geometry.x], radius=3.2, color='#222222', weight=1,
                        fill=True, fill_color='white', fill_opacity=1,
                        tooltip=str(s.get('stop_name', ''))).add_to(stop_group)

# Consolidated legend: transit modes, parking colorbar, walk shed, and the per-line stats table
present_groups = {}
for name in route_summary['route_long_name']:
    style = ROUTE_STYLE.get(name, DEFAULT_ROUTE)
    present_groups.setdefault(style['group'], []).append((name.replace('METRO ', ''), style['color']))


def _line_swatch(color):
    return (f'<span style="display:inline-block;width:16px;border-top:3px solid {color};'
            f'vertical-align:middle;margin-right:6px;"></span>')


def _box(fill, border='#999', extra=''):
    return (f'<span style="display:inline-block;width:14px;height:10px;background:{fill};'
            f'border:1px solid {border};{extra}vertical-align:middle;margin-right:6px;"></span>')


def _circle(fill, border='#999'):
    return (f'<span style="display:inline-block;width:11px;height:11px;border-radius:50%;'
            f'background:{fill};border:1px solid {border};vertical-align:middle;margin-right:6px;"></span>')


transit_legend = ''
for group in ['Light rail (LRT)', 'Bus rapid transit (BRT)', 'Arterial BRT (aBRT)', 'Other']:
    if group in present_groups:
        transit_legend += f'<div style="font-weight:600;margin:4px 0 1px;">{group}</div>'
        for name, color in present_groups[group]:
            transit_legend += f'<div>{_line_swatch(color)}{name}</div>'

gradient = 'linear-gradient(to right,' + ','.join(YLORRD) + ')'
stat_rows = ''
for line, row in walk_shed_summary.iterrows():
    stat_rows += (
        f'<tr><td style="text-align:left;padding-right:8px;">{line.replace("METRO ", "")} '
        f'({int(row["stops"])})</td><td>{row["shed_acres"]:,.0f}</td>'
        f'<td>{row["parking_pct_of_shed"]:.1f}%</td>'
        f'<td>{row["parking_pct_of_taxable_land"]:.1f}%</td>'
        f'<td>{row["parking_pct_of_land_value"]:.1f}%</td></tr>'
    )
legend_html = f'''
<div style="position:fixed;top:12px;right:12px;z-index:9999;background:rgba(255,255,255,0.95);
  padding:10px 12px;border-radius:6px;box-shadow:0 1px 5px rgba(0,0,0,.35);
  font:11px/1.45 Helvetica,Arial,sans-serif;color:#222;width:288px;">
  <div style="font-weight:bold;font-size:12px;margin-bottom:5px;">Parking near METRO transit &mdash; St. Paul</div>
  <div style="font-weight:bold;border-bottom:1px solid #bbb;margin-bottom:3px;">Transit</div>
  {transit_legend}
  <div>{_circle('white', '#222')}METRO stop</div>
  <div style="font-weight:bold;border-bottom:1px solid #bbb;margin:6px 0 3px;">Parking lots</div>
  <div style="height:11px;width:100%;background:{gradient};border:1px solid #999;"></div>
  <div style="display:flex;justify-content:space-between;font-size:10px;">
    <span>$0</span><span>$500K</span><span>$1M</span><span>$1.5M</span><span>$2M+</span></div>
  <div style="font-size:10px;color:#555;margin-top:1px;">taxed lots &mdash; underlying land value / acre</div>
  <div style="margin-top:3px;">{_box('#c0c0c0', '#9a9a9a')}Tax-exempt land (colleges, hospitals, churches, city)</div>
  <div style="font-weight:bold;border-bottom:1px solid #bbb;margin:6px 0 3px;">Walk shed</div>
  <div>{_box('#dbe4ee', '#5b7fa6')}10-min walk shed</div>
  <div>{_box('none', '#333', 'border-style:dashed;')}St. Paul boundary</div>
  <div style="font-weight:bold;border-bottom:1px solid #bbb;margin:6px 0 3px;">Parking share within walk shed</div>
  <table style="border-collapse:collapse;text-align:right;font-size:10px;">
    <tr style="font-weight:bold;border-bottom:1px solid #999;">
      <td style="text-align:left;padding-right:8px;">Line (stops)</td><td>Shed<br>acres</td>
      <td style="padding-left:6px;">% of<br>shed</td><td style="padding-left:6px;">% tax-<br>able land</td>
      <td style="padding-left:6px;">% land<br>value</td></tr>
    {stat_rows}
  </table>
</div>'''
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False, position='topright').add_to(m)
# Sit the layer-toggle box directly below the legend
m.get_root().header.add_child(folium.Element(
    '<style>.leaflet-control-layers{margin-top:555px !important;margin-right:12px !important;'
    'box-shadow:0 1px 5px rgba(0,0,0,.35) !important;}</style>'))
m.add_child(_ZoomFade(neighborhoods_layer, fade_start=14.0, fade_end=15.5))

# Frame the full St. Paul city boundary; at a wide window the network trails off
# toward Minneapolis (west) and the suburbs (east)
m.fit_bounds([[cminy, cminx], [cmaxy, cmaxx]])

# Standalone copy for viewing outside Jupyter (notebook must be trusted to render inline)
map_path = os.path.join(save_dir, 'transit_parking_map.html')
m.save(map_path)
print(f'{len(parking_in_sheds):,} OSM parking lots intersect the walk sheds '
      f'({int(is_exempt.sum())} on tax-exempt land)')
print(f'Interactive map saved to {map_path}')
m


In [ ]:
# Auto-export the map to a high-resolution PDF each run (headless Chrome -> 3x PNG -> 300-DPI PDF)
import subprocess
import shutil
from PIL import Image

png_path = os.path.join(save_dir, 'transit_parking_map.png')
pdf_path = os.path.join(save_dir, 'stpaul_transit_parking_map.pdf')

# folium maps need a browser engine to rasterize; find a Chromium-family browser
chrome = next((p for p in [
    '/Applications/Google Chrome.app/Contents/MacOS/Google Chrome',
    '/Applications/Chromium.app/Contents/MacOS/Chromium',
    shutil.which('google-chrome'), shutil.which('chromium'), shutil.which('chromium-browser')
] if p and os.path.exists(p)), None)

if chrome is None:
    print('No Chrome/Chromium found — skipping PDF export (install Chrome to enable).')
else:
    subprocess.run([chrome, '--headless', '--disable-gpu', '--hide-scrollbars',
                    '--force-device-scale-factor=2', '--window-size=1850,950',
                    '--virtual-time-budget=25000',
                    f'--screenshot={png_path}', f'file://{os.path.abspath(map_path)}'],
                   check=True, capture_output=True)
    img = Image.open(png_path).convert('RGB')
    # Crop the rendered frame. Values are DEVICE px (2x the --window-size above).
    #   trim_west  : px cut off the LEFT/west  -> bigger = less Minneapolis
    #                (too big starts cutting into Saint Anthony Park)
    #   trim_east  : px cut off the RIGHT/east -> bigger = less empty suburb
    #                (too big will cover the legend/checkbox)
    trim_west, trim_east = 300, 0
    img = img.crop((trim_west, 0, img.width - trim_east, img.height))
    img.save(pdf_path, 'PDF', resolution=300.0)
    w, h = img.size
    print(f'Exported high-res PDF: {pdf_path} ({w}x{h}px = {w/300:.1f}x{h/300:.1f} in @300dpi)')
img